1) Setup: MAPE + detectar colunas de Warengruppe

In [1]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import mean_absolute_error, r2_score



2) Load Dataset

In [2]:
df_train = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/03_cylindical/train_data.csv", encoding = "utf-8")
df_val = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/03_cylindical/val_data.csv", encoding = "utf-8")

df_train['Datum'] = pd.to_datetime(df_train['Datum'], format='%Y-%m-%d')
df_val['Datum'] = pd.to_datetime(df_val['Datum'], format='%Y-%m-%d')

In [3]:
# target column
TARGET = 'Umsatz'

# columns that should NOT go into the model
cols_drop = ['id', TARGET, 'Datum']

X_train = df_train.drop(columns=cols_drop)
y_train = df_train[TARGET]
X_val = df_val.drop(columns=cols_drop)
y_val = df_val[TARGET]

In [4]:

def mape_safe(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100

def get_warengruppe_series(df):
    # caso 1: coluna única já existe
    if "Warengruppe" in df.columns:
        return df["Warengruppe"].astype(int)

    # caso 2: one-hot (Warengruppe_*)
    wg_cols = [c for c in df.columns if c.startswith("Warengruppe_")]
    prefix = "Warengruppe_"

    # caso 3: one-hot (Group_*)
    if not wg_cols:
        wg_cols = [c for c in df.columns if c.startswith("Group_")]
        prefix = "Group_"

    if not wg_cols:
        raise ValueError("Warengruppe not found neither as a column nor as one-hot (Warengruppe_*/Group_*).")

    return df[wg_cols].idxmax(axis=1).str.replace(prefix, "", regex=False).astype(int)



3) Tuning: Random Search no val

In [5]:
param_dist = {
    "n_estimators": [300, 600, 1000, 1500, 2500],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "max_depth": [3, 4, 5, 6, 7, 8],
    "min_child_weight": [1, 3, 5, 10],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0.0, 0.05, 0.1, 0.2],
    "reg_alpha": [0.0, 1e-4, 1e-3, 1e-2],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0],
}

best_mape = np.inf
best_params = None
best_model = None

n_iter = 60

for i, params in enumerate(ParameterSampler(param_dist, n_iter=n_iter, random_state=42), start=1):
    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
        tree_method="hist",   
        **params
    )

    model.fit(X_train, y_train)

    pred_val = model.predict(X_val)
    mape_val = mape_safe(y_val, pred_val)

    if mape_val < best_mape:
        best_mape = mape_val
        best_params = params
        best_model = model
        print(f"[{i}/{n_iter}] NEW BEST  MAPE_VAL={best_mape:.3f}  params={best_params}")

print("\nBest VAL MAPE:", best_mape)
print("Best params:", best_params)


[1/60] NEW BEST  MAPE_VAL=21.378  params={'subsample': 1.0, 'reg_lambda': 0.5, 'reg_alpha': 0.01, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.02, 'gamma': 0.0, 'colsample_bytree': 0.8}
[2/60] NEW BEST  MAPE_VAL=20.646  params={'subsample': 1.0, 'reg_lambda': 5.0, 'reg_alpha': 0.001, 'n_estimators': 2500, 'min_child_weight': 10, 'max_depth': 5, 'learning_rate': 0.01, 'gamma': 0.05, 'colsample_bytree': 0.8}
[5/60] NEW BEST  MAPE_VAL=20.016  params={'subsample': 1.0, 'reg_lambda': 5.0, 'reg_alpha': 0.0001, 'n_estimators': 1000, 'min_child_weight': 10, 'max_depth': 7, 'learning_rate': 0.01, 'gamma': 0.0, 'colsample_bytree': 0.8}

Best VAL MAPE: 20.01637103643652
Best params: {'subsample': 1.0, 'reg_lambda': 5.0, 'reg_alpha': 0.0001, 'n_estimators': 1000, 'min_child_weight': 10, 'max_depth': 7, 'learning_rate': 0.01, 'gamma': 0.0, 'colsample_bytree': 0.8}


3) Evaluation (train/val) + MAPE by Warengruppe on VAL

In [8]:
y_pred_train = best_model.predict(X_train)
y_pred_val   = best_model.predict(X_val)

print('--- TRAIN ---')
print(f"MAE  : {mean_absolute_error(y_train, y_pred_train):,.2f}")
print(f"R²   : {r2_score(y_train, y_pred_train):,.3f}")
print(f"MAPE : {mape_safe(y_train, y_pred_train):,.2f}%")

print('\n--- VAL ---')
print(f"MAE  : {mean_absolute_error(y_val, y_pred_val):,.2f}")
print(f"R²   : {r2_score(y_val, y_pred_val):,.5f}")
print(f"MAPE : {mape_safe(y_val, y_pred_val):,.2f}%")


--- TRAIN ---
MAE  : 24.27
R²   : 0.928
MAPE : 14.33%

--- VAL ---
MAE  : 33.98
R²   : 0.83048
MAPE : 20.02%


In [7]:
df_val_eval = df_val.copy()
df_val_eval["y_true"] = y_val.to_numpy()
df_val_eval["y_pred"] = y_pred_val

mape_by_group = (
    df_val_eval.groupby("Warengruppe", as_index=False)
    .apply(lambda g: mape_safe(g["y_true"], g["y_pred"]))
    .rename(columns={None: "MAPE"})
    .sort_values("Warengruppe")
)

print("MAPE by Warengruppe (VAL):")
print(mape_by_group)

print("\nMacro-MAPE:", mape_by_group["MAPE"].mean())
print("Micro-MAPE:", mape_safe(df_val_eval["y_true"], df_val_eval["y_pred"]))



MAPE by Warengruppe (VAL):
   Warengruppe       MAPE
0            1  20.242397
1            2  15.644704
2            3  18.668442
3            4  24.647419
4            5  18.014223
5            6  38.278643

Macro-MAPE: 22.582638050287333
Micro-MAPE: 20.01637103643652


/tmp/ipykernel_33920/3497468075.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mape_safe(g["y_true"], g["y_pred"]))


4) Treino final (train+val) + salvar CSV do TEST + (opcional) MAPE por grupo no TEST

In [8]:
TARGET = "Umsatz"
cols_drop = ["id", TARGET, "Datum"]

df_test = pd.read_csv("/workspaces/bakery_prediction/0_DataPreparation/Split_data/03_cylindical/test_data.csv", encoding="utf-8")
df_test["Datum"] = pd.to_datetime(df_test["Datum"], format="%Y-%m-%d")

X_test = df_test.drop(columns=cols_drop)

# makes sure the columns have the same order as the training data
X_test = X_test.reindex(columns=X_train.columns)

y_pred_test = best_model.predict(X_test)

df_test["Umsatz_Predicted"] = y_pred_test

In [9]:
from xgboost import XGBRegressor

X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)

final_xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    **best_params   # <-- dict w best params
)

final_xgb.fit(X_trainval, y_trainval)

y_pred_test = final_xgb.predict(X_test)

df_test_out = df_test.copy()
df_test_out["Umsatz_Predicted"] = y_pred_test
out_path = "/workspaces/bakery_prediction/2_BaselineModel/03_XGB/predictions/xgb_tuned_predictions.csv"
df_test_out[["id", "Umsatz_Predicted"]].to_csv(out_path, index=False)
print("Salved to:", out_path)


Salved to: /workspaces/bakery_prediction/2_BaselineModel/03_XGB/predictions/xgb_tuned_predictions.csv
